In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import abc, gc, os, random, re, sys, time, traceback
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import stable_baselines3 as sb
from stable_baselines3.common.env_util import make_vec_env
from typing import SupportsFloat, Any
import gymnasium as gym
from gymnasium.core import ActType, ObsType

import faiss
import sklearn
import tqdm
import chime

import torch
from torch.nn import functional as F
from torch.utils import data
from torch import optim

In [4]:
from data_loaders import KuaiRecData, commit_dataset, CommitedDataset, JaggedCommittedDataset, commit_dataset_jagged
from dl_trainer import Trainer, sequence_collate, create_sequence_collator
from rs_algorithm.models import AdaptiveRepresentation
from rs_algorithm.models import TwoTowerRecommender, ThreeTowerRecommenderV2, ConcatInteractionHistoryRepresentationWithUserEmbedInitialization
from rs_algorithm.models.user_models.user_02_sim import DynamicPreferenceUserModel, norm_dot_product_based_relavance_calculation, SimUserModel

In [5]:
data_model = KuaiRecData(root='../data/KuaiRec2.0', initial_specimen='smaller')

In [6]:
d_embed = 128
n_users, d_user_feats = data_model.ratings['user_id'].nunique(), data_model.user_features.shape[1]
n_items, d_item_feats = data_model.ratings['item_id'].nunique(), data_model.item_features.shape[1]

uid2idx = {v: i for i, v in enumerate(np.unique(data_model.ratings['user_id']))}
user_model = AdaptiveRepresentation(n_users, d_user_feats, d_hidden=512, d_embed=d_embed, id2idx=uid2idx)
iid2idx = {v: i for i, v in enumerate(np.unique(data_model.ratings['item_id']))}
item_model = AdaptiveRepresentation(n_items, d_item_feats, d_hidden=512, d_embed=d_embed, id2idx=iid2idx)
s1_embed_model = TwoTowerRecommender(user_model, item_model)

Trainer(s1_embed_model, None).load_model('model_id_trained_embedder_v3_128.pt')
s1_embed_model

TwoTowerRecommender(
  (user_model): AdaptiveRepresentation(
    (embedder): Sequential(
      (0): Embedding(1411, 128)
      (1): Flatten(start_dim=1, end_dim=-1)
    )
    (adaptor): Sequential(
      (0): Linear(in_features=1582, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=512, bias=True)
      (3): ReLU()
      (4): Linear(in_features=512, out_features=512, bias=True)
      (5): ReLU()
      (6): Dropout(p=0.2, inplace=False)
      (7): Linear(in_features=512, out_features=128, bias=True)
    )
    (alpha): Sequential(
      (0): Linear(in_features=1582, out_features=1, bias=True)
      (1): Sigmoid()
    )
    (bias): Embedding(1411, 1)
  )
  (item_model): AdaptiveRepresentation(
    (embedder): Sequential(
      (0): Embedding(3327, 128)
      (1): Flatten(start_dim=1, end_dim=-1)
    )
    (adaptor): Sequential(
      (0): Linear(in_features=429, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, 

In [7]:
for mod in (user_model, item_model):
    for par in mod.parameters():
        par.requires_grad = False

history_model = ConcatInteractionHistoryRepresentationWithUserEmbedInitialization(d_embed, 512)
s2p_recommender = ThreeTowerRecommenderV2(d_embed, user_model, item_model, history_model)

Trainer(s2p_recommender, None).load_model('model_recommender_v6.pt')
s2p_recommender

ThreeTowerRecommenderV2(
  (user_model): AdaptiveRepresentation(
    (embedder): Sequential(
      (0): Embedding(1411, 128)
      (1): Flatten(start_dim=1, end_dim=-1)
    )
    (adaptor): Sequential(
      (0): Linear(in_features=1582, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=512, bias=True)
      (3): ReLU()
      (4): Linear(in_features=512, out_features=512, bias=True)
      (5): ReLU()
      (6): Dropout(p=0.2, inplace=False)
      (7): Linear(in_features=512, out_features=128, bias=True)
    )
    (alpha): Sequential(
      (0): Linear(in_features=1582, out_features=1, bias=True)
      (1): Sigmoid()
    )
    (bias): Embedding(1411, 1)
  )
  (item_model): AdaptiveRepresentation(
    (embedder): Sequential(
      (0): Embedding(3327, 128)
      (1): Flatten(start_dim=1, end_dim=-1)
    )
    (adaptor): Sequential(
      (0): Linear(in_features=429, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=5

In [8]:
index = faiss.IndexFlatIP(d_embed)
idxToIid = {}
with torch.no_grad():
    for idx, iid in enumerate(data_model.ratings['item_id'].unique()):
        item_model.eval()
        iem, *_ = item_model(
            torch.tensor([iid], dtype=torch.int64),
            torch.tensor(data_model.get_item_features(iid)[None, :], dtype=torch.float32))
        index.add(iem.numpy())
        idxToIid[idx] = iid


def get_relevant_items(vector, n=7):
    indices = index.search(vector, n)[1]
    return np.vectorize(idxToIid.get)(indices.flatten())


get_relevant_items(np.random.rand(1, d_embed))

array([ 600, 6787, 6222,  250, 4416, 8524, 2343])

In [9]:
def get_semi_relevant_items(vector, n=3):
    indices = index.search(vector, 1500 + n)[1]
    return np.vectorize(idxToIid.get)(indices[:, 1500:].flatten())


get_semi_relevant_items(np.random.rand(1, d_embed))

array([ 625, 5762, 4684])

In [10]:
def random_perpendicular(v):
    shape = v.shape
    v = v.flatten()
    v = np.asarray(v)
    rand = np.random.randn(*v.shape)  # random vector
    proj = np.dot(rand, v) / np.dot(v, v) * v  # projection of rand onto v
    perp = rand - proj  # subtract projection → perpendicular component
    return perp.reshape(shape)


def get_irrelevant_items(vector, n=7):
    vector = random_perpendicular(vector)
    return get_relevant_items(vector, n)


get_irrelevant_items(np.random.rand(1, d_embed))

array([ 600, 6787, 6222, 2590, 2343,  723,  250])

In [11]:
def get_random_items(n=2):
    indices = np.random.randint(0, n_items, size=n)
    return np.vectorize(idxToIid.get)(indices)


get_random_items()

array([ 835, 7758])

In [12]:
_e = np.random.rand(1, d_embed)
_r = get_relevant_items(_e, 1)
_s = get_semi_relevant_items(_e, 1)
_i = get_irrelevant_items(_e, 1)
_x = get_random_items(1)

with torch.no_grad():
    for _f in (_r, _s, _i, _x):
        iem, *_ = item_model(
            torch.tensor(_f, dtype=torch.int64),
            torch.tensor(data_model.get_item_features(_f), dtype=torch.float32))
        print((F.normalize(torch.tensor(_e), dim=1) * F.normalize(iem, dim=1)).sum(dim=1))

tensor([0.1053], dtype=torch.float64)
tensor([-0.0034], dtype=torch.float64)
tensor([0.1242], dtype=torch.float64)
tensor([0.0014], dtype=torch.float64)


In [13]:
with torch.no_grad():
    uids = data_model.ratings['user_id'].unique()
    user_model.eval()
    uem, _ = user_model(
        torch.tensor(uids, dtype=torch.int64),
        torch.tensor(data_model.get_user_features(uids), dtype=torch.float32))
    print(uem.shape)
user_embed_mean = uem.mean(dim=1).mean().numpy()
user_embed_std = uem.std(dim=1).mean().numpy()
print("Mean", user_embed_mean, "Std", user_embed_std)


def norm_press_preference(preference):
    preference = (preference - preference.mean()) / preference.std()
    return preference * user_embed_std + user_embed_mean

torch.Size([1411, 128])
Mean 25.018114 Std 463.4131


In [14]:
class SSREnvironment(gym.Env):
    def __init__(self, n_action=(70, 20, 10), max_steps=500, max_history_len=100, metrics_window=10):
        self.n_action = n_action
        self.max_steps = max_steps
        self.max_history_len = max_history_len
        self.metrics_window = metrics_window

        self.step_count = 0

        self.action_space = gym.spaces.Discrete(np.sum(n_action))
        self.observation_space = gym.spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(d_embed + 1 + 4,),  # embeds, bias and 4 internal state monitorable metrics
            dtype=np.float32
        )

    def reset(self, seed=None, options=None) -> tuple[ObsType, dict[str, Any]]:
        super().reset(seed=seed)

        uid = data_model.ratings['user_id'].sample(1).iloc[0]
        uft = data_model.get_user_features(uid)
        uid = torch.tensor([uid], dtype=torch.int64)
        uft = torch.tensor(uft[None, :], dtype=torch.float32)
        with torch.no_grad():
            user_model.eval()
            uem, ubs = user_model(uid, uft)
            uem, ubs = uem.cpu().numpy(), ubs.cpu().numpy()
        self.user_state = {'uid': uid, 'uft': uft}
        self.history_state = {'hid': [], 'hcl': []}

        self.user = SimUserModel(
            uem, ubs,
            relevance_calculator=norm_dot_product_based_relavance_calculation,
            preference_update_manager=norm_press_preference,
        )

        self.today = datetime.now()

        self.recent_click_rate = 0
        self.similarity_of_recent_items = 0
        self.diversity_score = 0
        self.session_length = 0

        return self._get_obs(), {}

    def step(self, action: ActType) -> tuple[ObsType, SupportsFloat, bool, bool, dict[str, Any]]:
        #
        if not self.user.can_interact_today(self.today):
            self.today += timedelta(days=1)
            self.session_length = 0
            return self._get_obs(), -1, False, False, self.user.get_internal_representation()

        uid, uft = self.user_state['uid'], self.user_state['uft']
        hid, hcl = self.history_state['hid'], self.history_state['hcl']
        hft = torch.tensor(data_model.get_item_features(hid)[None], dtype=torch.float32)
        hid = torch.tensor([hid], dtype=torch.int64)
        hcl = torch.tensor([hcl], dtype=torch.float32)
        msk = torch.tensor([[True] * hft.shape[1]], dtype=torch.bool)
        lns = torch.tensor([hft.shape[1]], dtype=torch.int64)
        with torch.no_grad():
            pem, (_, pbs) = s2p_recommender(uid, uft, hid, hft, hcl, msk, lns)
            pem, pbs = pem.cpu().numpy(), pbs.cpu().numpy()

        n_relevant_item_ids, n_semi_relevant_item_ids, n_random_item_ids = self.n_action
        relevant_item_ids = get_relevant_items(pem, n_relevant_item_ids)
        semi_relevant_item_ids = get_semi_relevant_items(pem, n_semi_relevant_item_ids)
        random_item_ids = get_random_items(n_random_item_ids)
        item_ids = np.concatenate((relevant_item_ids, semi_relevant_item_ids, random_item_ids))

        iid = item_ids[action]
        ift = data_model.get_item_features(iid)
        with torch.no_grad():
            item_model.eval()
            iem, ibs = item_model(
                torch.tensor([iid], dtype=torch.int64),
                torch.tensor(ift[None, :], dtype=torch.float32))
            iem, ibs = iem.cpu().numpy(), ibs.cpu().numpy()

        did_i_click = self.user.react(self.today, None, iem, ibs)

        self.history_state['hid'].append(iid)
        self.history_state['hcl'].append(did_i_click)
        if len(self.history_state['hid']) > self.max_history_len:
            self.history_state['hid'].pop(0)
            self.history_state['hcl'].pop(0)

        reward = (
                float(did_i_click)
                - 2 * (not self.user.can_interact_today(self.today))
                - 10 * self.user.is_gone_for_good()
                - 0.05 * self.user.fatigue
                - 0.05 * self.user.discontent
        )
        # TODO: Add another penalty if the user is fatigued
        # TODO: Add another reward for diversity
        # TODO: Add another penalty for repetition

        self.step_count += 1

        if hcl.shape[1] > 0:
            self.recent_click_rate = hcl[:, -self.metrics_window:].mean().item()
        if hcl.shape[1] > 1:
            hem, _ = item_model(hid[0, -self.metrics_window:], hft[0, -self.metrics_window:])
            self.similarity_of_recent_items = (F.normalize(hem[:-1], dim=1) * F.normalize(hem[1:], dim=1)).sum(dim=1).mean().item()
            self.diversity_score = 1 - self.similarity_of_recent_items

        self.session_length += 1

        terminated = self.user.is_gone_for_good()
        truncated = self.step_count >= self.max_steps

        return self._get_obs(), reward, terminated, truncated, self.user.get_internal_representation()

    def _get_obs(self):
        return np.hstack([
            self.user._state_vector['preference'].flatten(),
            np.asarray([
                (self.user._state_vector['preference_bias'] or [0.0])[0],
                self.recent_click_rate,
                self.similarity_of_recent_items,
                self.diversity_score,
                self.session_length,
            ], dtype=np.float32),
        ])


In [15]:
env = make_vec_env(SSREnvironment, n_envs=8)
model = sb.DQN("MlpPolicy", env, verbose=1, learning_rate=1e-3, buffer_size=10000, learning_starts=100, batch_size=32)
model

Using cuda device


In [16]:
model.learn(total_timesteps=3000)

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 32.2     |
|    ep_rew_mean      | 2.01     |
|    exploration_rate | 0.139    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 112      |
|    time_elapsed     | 2        |
|    total_timesteps  | 272      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 113      |
|    n_updates        | 5        |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 34.5     |
|    ep_rew_mean      | 1.8      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 116      |
|    time_elapsed     | 2        |
|    total_timesteps  | 328      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 73.4     |
|    n_updates      

In [17]:
model.save("temp/checkpoints/dqn_system_2")

In [409]:
test_env = SSREnvironment()
obs, _ = test_env.reset()

In [410]:
for _ in range(20):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = test_env.step(action)
    print(reward)
    if done or truncated: break

0.9945
-0.003
-0.006500000000000001
-0.004
-0.005500000000000001
-0.008000000000000002
-0.010500000000000002
-0.013000000000000001
0.9845
0.982
-0.0205
-0.023
0.9745
-0.028000000000000004
-0.030500000000000006
-0.03300000000000001
-0.03550000000000001
0.962
-0.040500000000000015
-0.04300000000000002
